# Transferability of established disaster-recovery metrics to Samar–Leyte

## Experimental objective

This notebook tests whether three established nighttime lights methods can be transferred to Haiyan using the available Samar–Leyte VNP46A2 record, and whether their conclusions change after reliability qualification.

The comparison follows two stages:

1. **Published-method replication:** preserve each paper’s baseline, temporal aggregation, gap treatment, and recovery definition as closely as the available inputs allow.
2. **Reliability-qualified replication:** recalculate comparable outputs using fresh `MQF == 0` DNB-BRDF observations, the fixed GHSL G7 mask, and spatial-completeness thresholds of 50% and 60%.

The methods are not treated as interchangeable:

| Method | Published recovery concept | Main implementation |
|---|---|---|
| [Román et al. (2019)](https://doi.org/10.1371/journal.pone.0218883) | Relative NTL recovery and net days without electricity | Four-day aggregation; three approximately 60-day recovery stages |
| [Mo et al. (2025)](https://doi.org/10.1088/2634-4505/ade474) | Tropical-cyclone blackout duration | Three-month baseline; lower-tail blackout detection; next-available gap filling; two consecutive baseline days |
| [Chakraborty and Stokes (2023)](https://doi.org/10.1016/j.rse.2023.113818) | Forecast-relative change severity and recovery | Gap-filled area-weighted NTL; 30-day moving average; neural-network forecasts trained on at least three pre-change years |

## Replication boundary

These are **metric-equivalent replications**, not complete reproductions of every original spatial input.

- Román et al. used Collection 1.1 and a Puerto Rico-specific 30 m Black Marble HD product. The code reproduces the published recovery and NDWE equations at the native VNP46A2 scale.
- Mo et al. processed GHSL clusters at 5 km. The code preserves the temporal and blackout rules but applies them to the fixed Samar–Leyte G7 support.
- Chakraborty and Stokes require at least three pre-change years. Exact application to Haiyan is structurally impossible because VIIRS begins in 2012. The code records this as a transferability failure instead of shortening the training period and mislabelling the result as an exact replication.

In [28]:
pip install rioxarray

Note: you may need to restart the kernel to use updated packages.


In [29]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

from rasterio.enums import Resampling

import plotly.express as px

from IPython.display import display


warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A1_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A1.zarr"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

GHSL_CANDIDATES = [
    VNP46_DIR / "GHSL_SMOD_E2015.tif",
    DATA_DIR / "ghsl" / "GHSL_SMOD_E2015.tif",
]

GHSL_PATH = next(
    (path for path in GHSL_CANDIDATES if path.exists()),
    GHSL_CANDIDATES[0],
)

NGCP_CANDIDATES = [
    DATA_DIR / "NGCP_hourly_load.xlsx",
    PROJECT_DIR / "Datasets" / "NGCP_hourly_load.xlsx",
    DATA_DIR / "NGCP" / "NGCP_hourly_load.xlsx",
]

NGCP_PATH = next(
    (path for path in NGCP_CANDIDATES if path.exists()),
    NGCP_CANDIDATES[0],
)

NGCP_SHEET = "LEY-SAM HOURLY LOAD 2013-2024"

# ------------------------------------------------------------
# Event and analysis settings
# ------------------------------------------------------------

EVENT_DATE = pd.Timestamp("2013-11-08")

ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)

ROMAN_BASELINE_START = EVENT_DATE - pd.Timedelta(days=4)
ROMAN_BASELINE_END = EVENT_DATE - pd.Timedelta(days=1)
ROMAN_POST_END = EVENT_DATE + pd.Timedelta(days=179)

ROMAN_BLOCK_DAYS = 4
ROMAN_STAGE_DAYS = 60
ROMAN_STAGE_COUNT = 3
ROMAN_BLOCKS_PER_STAGE = 15

# Reliability-qualified implementation
GHSL_G7_CLASSES = (23, 30)
RQ_MIN_SPATIAL_COVERAGE = 10.0

# ------------------------------------------------------------
# Bands
# ------------------------------------------------------------

DNB_BAND = "DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------

PLOT_TEMPLATE = "plotly_white"
EVENT_LINE_COLOR = "#2563EB"

SERIES_COLORS = {
    "Román-style NTL": "#D97706",
    "Reliability-qualified NTL": "#059669",
    "NGCP 1 AM load": "#334155",
}

print("VNP46A2:", A2_ZARR_PATH)
print("GHSL:", GHSL_PATH)
print("NGCP:", NGCP_PATH)

VNP46A2: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/VNP46/processed/Haiyan_VNP46A2.zarr
GHSL: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/ghsl/GHSL_SMOD_E2015.tif
NGCP: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/NGCP/NGCP_hourly_load.xlsx


In [30]:
for label, path in {
    "VNP46A2": A2_ZARR_PATH,
    "GHSL": GHSL_PATH,
    "NGCP": NGCP_PATH,
}.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{label} input was not found:\n{path}"
        )


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(
            f"No `date` variable found. Variables: {list(ds.variables)}"
        )

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]

    dates = pd.DatetimeIndex(
        pd.to_datetime(ds["date"].values)
    ).normalize()

    ds = ds.assign_coords(
        date=(observation_dim, dates.values)
    )

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


a2 = standardise_date_dimension(
    open_zarr_safely(A2_ZARR_PATH)
)

a2 = a2.sel(
    date=slice(ANALYSIS_START, ANALYSIS_END)
)

a2 = a2.rio.set_spatial_dims(
    x_dim="x",
    y_dim="y",
    inplace=False,
)

if a2.rio.crs is None and "spatial_ref" in a2.variables:
    spatial_attrs = a2["spatial_ref"].attrs

    stored_crs = (
        spatial_attrs.get("crs_wkt")
        or spatial_attrs.get("spatial_ref")
    )

    if stored_crs is not None:
        a2 = a2.rio.write_crs(
            stored_crs,
            inplace=False,
        )

if a2.rio.crs is None:
    x_min = float(a2["x"].min())
    x_max = float(a2["x"].max())
    y_min = float(a2["y"].min())
    y_max = float(a2["y"].max())

    if (
        -180 <= x_min <= 180
        and -180 <= x_max <= 180
        and -90 <= y_min <= 90
        and -90 <= y_max <= 90
    ):
        a2 = a2.rio.write_crs(
            "EPSG:4326",
            inplace=False,
        )
    else:
        raise ValueError(
            "The VNP46A2 CRS could not be recovered."
        )

missing_bands = [
    band
    for band in [DNB_BAND, MQF_BAND]
    if band not in a2.data_vars
]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available bands: {list(a2.data_vars)}"
    )

dnb = a2[DNB_BAND].astype("float32")
mqf = a2[MQF_BAND]

dnb, mqf = xr.align(
    dnb,
    mqf,
    join="inner",
)

SPATIAL_DIMS = ("y", "x")

print("A2 dimensions:", dict(a2.sizes))
print(
    "Dates:",
    pd.Timestamp(a2["date"].values.min()).date(),
    "to",
    pd.Timestamp(a2["date"].values.max()).date(),
)

A2 dimensions: {'date': 546, 'y': 674, 'x': 473}
Dates: 2013-05-12 to 2014-11-08


In [31]:
# ------------------------------------------------------------
# 1. Román-style signal
# ------------------------------------------------------------
# DNB-BRDF as supplied, without the RQ1 MQF/GHSL filters.

roman_daily = dnb.mean(
    dim=SPATIAL_DIMS,
    skipna=True,
).to_series()

roman_daily.index = pd.to_datetime(
    roman_daily.index
).normalize()

roman_daily.name = "ntl"


# ------------------------------------------------------------
# 2. Reliability-qualified signal
# ------------------------------------------------------------

ghsl = rxr.open_rasterio(
    GHSL_PATH,
    masked=True,
)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(
        band=0,
        drop=True,
    )

viirs_template = dnb.isel(
    date=0,
    drop=True,
)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
)

ghsl_viirs = ghsl_viirs.assign_coords(
    x=viirs_template["x"],
    y=viirs_template["y"],
)

g7_mask = ghsl_viirs.isin(
    GHSL_G7_CLASSES
).fillna(False)

fresh_dnb = dnb.where(mqf == 0)
fresh_g7 = fresh_dnb.where(g7_mask)

g7_pixel_count = int(
    np.asarray(
        g7_mask.sum().compute().values
    ).item()
)

if g7_pixel_count == 0:
    raise ValueError(
        "The GHSL G7 mask contains no VIIRS pixels."
    )

daily_valid_pixels = (
    fresh_dnb
    .notnull()
    .where(g7_mask, False)
    .sum(dim=SPATIAL_DIMS)
)

daily_spatial_coverage = (
    100.0
    * daily_valid_pixels
    / g7_pixel_count
)

rq_daily_da = fresh_g7.mean(
    dim=SPATIAL_DIMS,
    skipna=True,
)

rq_daily_da = rq_daily_da.where(
    daily_spatial_coverage
    >= RQ_MIN_SPATIAL_COVERAGE
)

rq_daily = rq_daily_da.to_series()

rq_daily.index = pd.to_datetime(
    rq_daily.index
).normalize()

rq_daily.name = "ntl"


signal_summary = pd.DataFrame(
    {
        "signal": [
            "Román-style NTL",
            "Reliability-qualified NTL",
        ],
        "spatial_filter": [
            "None",
            "GHSL G7",
        ],
        "quality_filter": [
            "None",
            "MQF == 0",
        ],
        "minimum_spatial_coverage": [
            None,
            RQ_MIN_SPATIAL_COVERAGE,
        ],
        "available_daily_observations": [
            int(roman_daily.notna().sum()),
            int(rq_daily.notna().sum()),
        ],
    }
)

display(signal_summary)

,signal,spatial_filter,quality_filter,minimum_spatial_coverage,available_daily_observations
0,Román-style NTL,None,None,NaN,435
1,Reliability-qualified NTL,GHSL G7,MQF == 0,10.0,283


In [32]:
# ============================================================
# 3. CLEAN VNP46A2 AND CONSTRUCT THE TWO SIGNALS
# ============================================================

# These are validity controls applied to both implementations.
KNOWN_NTL_FILL_VALUES = (
    -9999.0,
    65535.0,
    6553.5,
)

MIN_VALID_RADIANCE = 0.0
MIN_BASELINE_RADIANCE = 1.0
MIN_DAYS_PER_4DAY_COMPOSITE = 2


def clean_ntl_radiance(values):
    """
    Remove non-finite values, stored fill values, and physically
    invalid negative radiance.

    This is basic input validation, not reliability qualification.
    """
    cleaned = values.astype("float32")

    cleaned = cleaned.where(
        np.isfinite(cleaned)
    )

    attribute_fill_values = [
        values.attrs.get("_FillValue"),
        values.attrs.get("missing_value"),
        values.encoding.get("_FillValue"),
    ]

    fill_values = list(
        KNOWN_NTL_FILL_VALUES
    )

    for fill_value in attribute_fill_values:
        if fill_value is not None:
            fill_values.append(fill_value)

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)

            cleaned = cleaned.where(
                cleaned != fill_value
            )
        except (TypeError, ValueError):
            continue

    cleaned = cleaned.where(
        cleaned >= MIN_VALID_RADIANCE
    )

    return cleaned


raw_dnb = a2[DNB_BAND].astype("float32")
mqf = a2[MQF_BAND]

dnb = clean_ntl_radiance(raw_dnb)

dnb, mqf = xr.align(
    dnb,
    mqf,
    join="inner",
)

SPATIAL_DIMS = ("y", "x")


# ------------------------------------------------------------
# Check what was removed
# ------------------------------------------------------------

removed_value_count = (
    raw_dnb.notnull()
    & dnb.isnull()
).sum()

removed_value_count = int(
    np.asarray(
        removed_value_count.compute().values
    ).item()
)

print(
    "Removed fill/invalid observations:",
    f"{removed_value_count:,}",
)

print(
    "Clean DNB minimum:",
    float(dnb.min(skipna=True).compute()),
)

print(
    "Clean DNB maximum:",
    float(dnb.max(skipna=True).compute()),
)


# ------------------------------------------------------------
# Align GHSL to the VIIRS grid
# ------------------------------------------------------------

ghsl = rxr.open_rasterio(
    GHSL_PATH,
    masked=True,
)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(
        band=0,
        drop=True,
    )

viirs_template = dnb.isel(
    date=0,
    drop=True,
)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
)

ghsl_viirs = ghsl_viirs.assign_coords(
    x=viirs_template["x"],
    y=viirs_template["y"],
)

g7_mask = ghsl_viirs.isin(
    GHSL_G7_CLASSES
).fillna(False)


# ------------------------------------------------------------
# 1. Román-style cube
# ------------------------------------------------------------
# Clean DNB-BRDF without MQF, GHSL, or coverage qualification.

roman_cube = dnb


# ------------------------------------------------------------
# 2. Reliability-qualified cube
# ------------------------------------------------------------

fresh_g7 = dnb.where(
    (mqf == 0) & g7_mask
)

g7_pixel_count = int(
    np.asarray(
        g7_mask.sum().compute().values
    ).item()
)

if g7_pixel_count == 0:
    raise ValueError(
        "The GHSL G7 mask contains no VIIRS pixels."
    )

daily_valid_pixels = (
    fresh_g7
    .notnull()
    .sum(dim=SPATIAL_DIMS)
)

daily_spatial_coverage = (
    100.0
    * daily_valid_pixels
    / g7_pixel_count
)

rq_cube = fresh_g7.where(
    daily_spatial_coverage
    >= RQ_MIN_SPATIAL_COVERAGE
)


signal_summary = pd.DataFrame(
    {
        "signal": [
            "Román-style NTL",
            "Reliability-qualified NTL",
        ],
        "fill_values_removed": [
            True,
            True,
        ],
        "minimum_radiance": [
            MIN_VALID_RADIANCE,
            MIN_VALID_RADIANCE,
        ],
        "quality_filter": [
            "None",
            "MQF == 0",
        ],
        "spatial_filter": [
            "Baseline-lit pixels",
            "Baseline-lit GHSL G7 pixels",
        ],
        "daily_coverage_threshold": [
            None,
            RQ_MIN_SPATIAL_COVERAGE,
        ],
    }
)

display(signal_summary)

Removed fill/invalid observations: 0
Clean DNB minimum: 0.0
Clean DNB maximum: 361.5819396972656


,signal,fill_values_removed,minimum_radiance,quality_filter,spatial_filter,daily_coverage_threshold
0,Román-style NTL,True,0.0,None,Baseline-lit pixels,NaN
1,Reliability-qualified NTL,True,0.0,MQF == 0,Baseline-lit GHSL G7 pixels,10.0


In [33]:
# ============================================================
# 4. LOAD SAMAR–LEYTE NGCP 1 AM LOAD
# ============================================================

# The actual column headings are on Excel row 3:
# pandas uses zero-based indexing, therefore header=2.
ngcp_raw = pd.read_excel(
    NGCP_PATH,
    sheet_name=NGCP_SHEET,
    header=2,
)

ngcp_raw = ngcp_raw.dropna(
    axis=1,
    how="all",
)


def normalise_column_name(column):
    return (
        str(column)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
    )


column_lookup = {
    normalise_column_name(column): column
    for column in ngcp_raw.columns
}

# Identify the date column.
date_column = next(
    (
        original_column
        for normalised_column, original_column
        in column_lookup.items()
        if "date" in normalised_column
    ),
    None,
)

# Fallback: select the column containing the most valid dates.
if date_column is None:
    date_counts = {}

    for column in ngcp_raw.columns:
        candidate_dates = pd.to_datetime(
            ngcp_raw[column],
            errors="coerce",
        )

        plausible_dates = candidate_dates.between(
            "2012-01-01",
            "2025-12-31",
        )

        date_counts[column] = int(
            plausible_dates.sum()
        )

    date_column = max(
        date_counts,
        key=date_counts.get,
    )

    if date_counts[date_column] == 0:
        raise KeyError(
            "The NGCP date column could not be identified.\n"
            f"Available columns: {list(ngcp_raw.columns)}"
        )

# Identify Hour 1 / 1 AM.
hour_1_candidates = {
    "1",
    "1.0",
    "hour1",
    "hour01",
    "hr1",
    "hr01",
    "1am",
    "01am",
    "1:00",
    "01:00",
    "1:00:00",
    "01:00:00",
}

hour_1_column = next(
    (
        original_column
        for normalised_column, original_column
        in column_lookup.items()
        if normalised_column in hour_1_candidates
    ),
    None,
)

if hour_1_column is None:
    raise KeyError(
        "The NGCP Hour 1 column could not be identified.\n"
        f"Available columns after header correction: "
        f"{list(ngcp_raw.columns)}"
    )

ngcp_dates = pd.to_datetime(
    ngcp_raw[date_column],
    errors="coerce",
)

ngcp_daily = pd.DataFrame(
    {
        "date": ngcp_dates,
        "load": pd.to_numeric(
            ngcp_raw[hour_1_column],
            errors="coerce",
        ),
    }
)

ngcp_daily = (
    ngcp_daily
    .dropna(subset=["date", "load"])
    .assign(
        date=lambda frame: frame["date"].dt.normalize()
    )
    .drop_duplicates(subset="date")
    .set_index("date")
    .sort_index()
)

ngcp_load = ngcp_daily["load"].loc[
    ANALYSIS_START:ANALYSIS_END
]

if ngcp_load.empty:
    raise ValueError(
        "No Samar–Leyte NGCP observations were found "
        "inside the Haiyan analysis period."
    )

print("Date column:", repr(date_column))
print("Hour 1 column:", repr(hour_1_column))
print("NGCP observations:", len(ngcp_load))
print(
    "NGCP date range:",
    ngcp_load.index.min().date(),
    "to",
    ngcp_load.index.max().date(),
)

display(ngcp_load.head())

Date column: 'DATE'
Hour 1 column: 1
NGCP observations: 546
NGCP date range: 2013-05-12 to 2014-11-08


date
2013-05-12    167.0
2013-05-13    155.0
2013-05-14    160.0
2013-05-15    165.0
2013-05-16    160.0
Name: load, dtype: float64

In [34]:
# ============================================================
# 5. FOUR-DAY PIXEL COMPOSITES
# ============================================================

def four_day_pixel_composites(cube):
    """
    Construct non-overlapping four-day composites at each pixel.

    Temporal aggregation is performed before spatial aggregation.
    """
    cube = cube.sel(
        date=slice(
            ROMAN_BASELINE_START,
            ROMAN_POST_END,
        )
    )

    dates = pd.DatetimeIndex(
        pd.to_datetime(cube["date"].values)
    ).normalize()

    relative_days = (
        (dates - EVENT_DATE)
        / pd.Timedelta(days=1)
    ).astype("int64")

    blocks = np.floor_divide(
        relative_days,
        ROMAN_BLOCK_DAYS,
    ).astype("int64")

    cube = cube.assign_coords(
        block=("date", blocks)
    )

    composites = (
        cube.groupby("block")
        .mean(
            dim="date",
            skipna=True,
        )
    )

    observed_days = (
        cube.notnull()
        .groupby("block")
        .sum(dim="date")
    )

    expected_blocks = np.arange(
        -1,
        45,
    )

    composites = composites.reindex(
        block=expected_blocks
    )

    observed_days = observed_days.reindex(
        block=expected_blocks,
        fill_value=0,
    )

    # A four-day value supported by only one observation is unstable.
    composites = composites.where(
        observed_days
        >= MIN_DAYS_PER_4DAY_COMPOSITE
    )

    return composites, observed_days

In [35]:
# ============================================================
# 6. PIXEL-FIRST ROMÁN METRICS
# ============================================================

def calculate_roman_pixel_metrics(
    cube,
    method,
):
    """
    Apply Román recovery metrics at the native VIIRS pixel scale,
    then aggregate the results across Samar–Leyte.
    """
    composites, observed_days = (
        four_day_pixel_composites(cube)
    )

    composites = composites.compute()
    observed_days = observed_days.compute()

    baseline_ntl = composites.sel(
        block=-1,
        drop=True,
    )

    baseline_observed_days = observed_days.sel(
        block=-1,
        drop=True,
    )

    # Fixed baseline-lit support.
    baseline_mask = (
        np.isfinite(baseline_ntl)
        & (
            baseline_ntl
            >= MIN_BASELINE_RADIANCE
        )
        & (
            baseline_observed_days
            >= MIN_DAYS_PER_4DAY_COMPOSITE
        )
    )

    baseline_ntl = baseline_ntl.where(
        baseline_mask
    )

    baseline_pixel_count = int(
        np.asarray(
            baseline_mask.sum().values
        ).item()
    )

    if baseline_pixel_count == 0:
        raise ValueError(
            f"{method}: no valid baseline-lit pixels."
        )

    # --------------------------------------------------------
    # Four-day recovery trajectory
    # --------------------------------------------------------

    pixel_recovery = (
        100.0
        * composites
        / baseline_ntl
    )

    pixel_recovery = pixel_recovery.where(
        baseline_mask
    )

    supported_pixels = (
        pixel_recovery
        .notnull()
        .sum(dim=SPATIAL_DIMS)
    )

    spatial_support_pct = (
        100.0
        * supported_pixels
        / baseline_pixel_count
    )

    regional_recovery = (
        pixel_recovery
        .mean(
            dim=SPATIAL_DIMS,
            skipna=True,
        )
    )

    regional_ntl = (
        composites
        .where(baseline_mask)
        .mean(
            dim=SPATIAL_DIMS,
            skipna=True,
        )
    )

    mean_observed_days = (
        observed_days
        .where(baseline_mask)
        .mean(
            dim=SPATIAL_DIMS,
            skipna=True,
        )
    )

    trajectory = xr.Dataset(
        {
            "value": regional_ntl,
            "recovery_pct": regional_recovery,
            "observed_days": mean_observed_days,
            "supported_pixels": supported_pixels,
            "spatial_support_pct": spatial_support_pct,
        }
    ).to_dataframe().reset_index()

    trajectory["date_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            trajectory["block"]
            * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    trajectory["date_end"] = (
        trajectory["date_start"]
        + pd.Timedelta(days=3)
    )

    trajectory["method"] = method

    # --------------------------------------------------------
    # Three 60-day stages
    # --------------------------------------------------------

    stage_rows = []
    stage_ndwe_arrays = []

    for stage in range(
        1,
        ROMAN_STAGE_COUNT + 1,
    ):
        first_block = (
            (stage - 1)
            * ROMAN_BLOCKS_PER_STAGE
        )

        last_block = (
            stage
            * ROMAN_BLOCKS_PER_STAGE
            - 1
        )

        stage_composites = composites.sel(
            block=slice(
                first_block,
                last_block,
            )
        )

        stage_ntl = stage_composites.mean(
            dim="block",
            skipna=True,
        )

        stage_available_composites = (
            stage_composites
            .notnull()
            .sum(dim="block")
        )

        stage_recovery = (
            100.0
            * stage_ntl
            / baseline_ntl
        )

        stage_recovery = stage_recovery.where(
            baseline_mask
        )

        stage_ndwe = (
            1.0
            - stage_ntl / baseline_ntl
        ) * ROMAN_STAGE_DAYS

        stage_ndwe = stage_ndwe.where(
            baseline_mask
        )

        stage_ndwe_arrays.append(
            stage_ndwe
        )

        stage_supported_pixels = int(
            np.asarray(
                stage_recovery
                .notnull()
                .sum()
                .values
            ).item()
        )

        stage_rows.append(
            {
                "method": method,
                "stage": stage,
                "stage_start": (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=(stage - 1) * 60
                    )
                ),
                "stage_end": (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=stage * 60 - 1
                    )
                ),
                "NTL0": float(
                    baseline_ntl.mean(
                        dim=SPATIAL_DIMS,
                        skipna=True,
                    )
                ),
                "stage_NTL": float(
                    stage_ntl.mean(
                        dim=SPATIAL_DIMS,
                        skipna=True,
                    )
                ),
                "recovery_pct": float(
                    stage_recovery.mean(
                        dim=SPATIAL_DIMS,
                        skipna=True,
                    )
                ),
                "ndwe_days": float(
                    stage_ndwe.mean(
                        dim=SPATIAL_DIMS,
                        skipna=True,
                    )
                ),
                "baseline_pixels": (
                    baseline_pixel_count
                ),
                "stage_supported_pixels": (
                    stage_supported_pixels
                ),
                "stage_spatial_support_pct": (
                    100.0
                    * stage_supported_pixels
                    / baseline_pixel_count
                ),
                "mean_available_4day_composites": float(
                    stage_available_composites
                    .where(baseline_mask)
                    .mean(
                        dim=SPATIAL_DIMS,
                        skipna=True,
                    )
                ),
            }
        )

    total_ndwe = xr.concat(
        stage_ndwe_arrays,
        dim="stage",
    ).sum(
        dim="stage",
        skipna=False,
    )

    mean_total_ndwe = float(
        total_ndwe.mean(
            dim=SPATIAL_DIMS,
            skipna=True,
        )
    )

    stage_results = pd.DataFrame(
        stage_rows
    )

    stage_results[
        "total_ndwe_days"
    ] = mean_total_ndwe

    return (
        stage_results,
        trajectory,
        baseline_mask,
    )

In [36]:
# ============================================================
# 7. RUN ONLY THE TWO ROMÁN IMPLEMENTATIONS
# ============================================================

(
    roman_stage_results,
    roman_trajectory,
    roman_baseline_mask,
) = calculate_roman_pixel_metrics(
    roman_cube,
    "Román-style NTL",
)

(
    rq_stage_results,
    rq_trajectory,
    rq_baseline_mask,
) = calculate_roman_pixel_metrics(
    rq_cube,
    "Reliability-qualified NTL",
)

stage_results = pd.concat(
    [
        roman_stage_results,
        rq_stage_results,
    ],
    ignore_index=True,
)

ntl_trajectories = pd.concat(
    [
        roman_trajectory,
        rq_trajectory,
    ],
    ignore_index=True,
)

display(
    stage_results[
        [
            "method",
            "stage",
            "recovery_pct",
            "ndwe_days",
            "baseline_pixels",
            "stage_supported_pixels",
            "stage_spatial_support_pct",
            "mean_available_4day_composites",
            "total_ndwe_days",
        ]
    ].round(2)
)

ValueError: Reliability-qualified NTL: no valid baseline-lit pixels.

## Method 1 — Román et al. (2019)

Román et al. defined:

$ \text{Recovery}_i = \frac{\mathrm{NTL}_i}{\mathrm{NTL}_0}
\times 100 $

$\mathrm{NDWE} = \sum_{i=1}^{3} \left(1-\frac{\mathrm{NTL}_i}{\mathrm{NTL}_0}\right)D_i $

where $\mathrm{NTL}_0$ is pre-hurricane radiance, $\mathrm{NTL}_i$ is radiance during recovery stage $i$, and $D_i$ is the stage duration. The original study used three stages averaging approximately 60 days and four-day multi-date aggregation.

Customer Hours of Interruption is not reproduced because household counts spatially aligned with the current NTL support are unavailable:

$\mathrm{CHI} = \mathrm{NDWE}\times H\times24 $

NGCP load is analysed separately as an independent regional comparator. It is not substituted for household count \(H\).

In [ ]:
def four_day_composites(series):
    """
    Non-overlapping four-day mean composites anchored to Haiyan.

    Block -1: 4–7 November 2013
    Block  0: 8–11 November 2013
    Blocks 0–44: three 60-day recovery stages
    """
    calendar = pd.date_range(
        ROMAN_BASELINE_START,
        ROMAN_POST_END,
        freq="D",
    )

    series = series.copy()
    series.index = pd.to_datetime(
        series.index
    ).normalize()

    series = series.groupby(
        series.index
    ).mean()

    series = series.reindex(calendar)

    relative_day = (
        calendar - EVENT_DATE
    ).days

    block = np.floor_divide(
        relative_day,
        ROMAN_BLOCK_DAYS,
    )

    frame = pd.DataFrame(
        {
            "date": calendar,
            "block": block,
            "value": series.values,
        }
    )

    composites = (
        frame.groupby("block")
        .agg(
            value=("value", "mean"),
            observed_days=("value", "count"),
        )
        .reset_index()
    )

    composites = composites[
        composites["block"].between(-1, 44)
    ].copy()

    composites["date_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            composites["block"]
            * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    composites["date_end"] = (
        composites["date_start"]
        + pd.Timedelta(days=3)
    )

    return composites


def roman_metrics(composites, method):
    """Calculate stage recovery and NDWE."""
    baseline = composites.loc[
        composites["block"] == -1,
        "value",
    ]

    if baseline.empty or pd.isna(baseline.iloc[0]):
        raise ValueError(
            f"{method}: the pre-Haiyan four-day baseline is missing."
        )

    baseline_value = float(
        baseline.iloc[0]
    )

    if baseline_value <= 0:
        raise ValueError(
            f"{method}: NTL₀ must be greater than zero."
        )

    stage_rows = []

    for stage in range(1, 4):
        first_block = (
            (stage - 1)
            * ROMAN_BLOCKS_PER_STAGE
        )

        last_block = (
            stage
            * ROMAN_BLOCKS_PER_STAGE
            - 1
        )

        stage_data = composites[
            composites["block"].between(
                first_block,
                last_block,
            )
        ]

        stage_ntl = stage_data[
            "value"
        ].mean()

        recovery_pct = (
            100.0
            * stage_ntl
            / baseline_value
        )

        ndwe_days = (
            1.0
            - stage_ntl / baseline_value
        ) * ROMAN_STAGE_DAYS

        stage_rows.append(
            {
                "method": method,
                "stage": stage,
                "stage_start": (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=(stage - 1) * 60
                    )
                ),
                "stage_end": (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=stage * 60 - 1
                    )
                ),
                "NTL0": baseline_value,
                "stage_NTL": stage_ntl,
                "recovery_pct": recovery_pct,
                "ndwe_days": ndwe_days,
                "available_4day_composites": int(
                    stage_data["value"].notna().sum()
                ),
            }
        )

    stage_results = pd.DataFrame(
        stage_rows
    )

    stage_results["total_ndwe_days"] = (
        stage_results["ndwe_days"].sum()
    )

    recovery_trajectory = (
        composites.copy()
    )

    recovery_trajectory[
        "recovery_pct"
    ] = (
        100.0
        * recovery_trajectory["value"]
        / baseline_value
    )

    recovery_trajectory["method"] = method

    return stage_results, recovery_trajectory

In [ ]:
roman_composites = four_day_composites(
    roman_daily
)

rq_composites = four_day_composites(
    rq_daily
)

roman_stage_results, roman_trajectory = (
    roman_metrics(
        roman_composites,
        "Román-style NTL",
    )
)

rq_stage_results, rq_trajectory = (
    roman_metrics(
        rq_composites,
        "Reliability-qualified NTL",
    )
)

stage_results = pd.concat(
    [
        roman_stage_results,
        rq_stage_results,
    ],
    ignore_index=True,
)

ntl_trajectories = pd.concat(
    [
        roman_trajectory,
        rq_trajectory,
    ],
    ignore_index=True,
)

display(
    stage_results[
        [
            "method",
            "stage",
            "stage_start",
            "stage_end",
            "recovery_pct",
            "ndwe_days",
            "available_4day_composites",
            "total_ndwe_days",
        ]
    ].round(2)
)

,method,stage,stage_start,stage_end,recovery_pct,ndwe_days,available_4day_composites,total_ndwe_days
0,Román-style NTL,1,2013-11-08,2014-01-06,225.949997,-75.570000,15,-538.059998
1,Román-style NTL,2,2014-01-07,2014-03-07,272.899994,-103.739998,14,-538.059998
2,Román-style NTL,3,2014-03-08,2014-05-06,697.919983,-358.750000,15,-538.059998
3,Reliability-qualified NTL,1,2013-11-08,2014-01-06,93.150002,4.110000,14,-106.919998
4,Reliability-qualified NTL,2,2014-01-07,2014-03-07,163.279999,-37.970001,11,-106.919998
5,Reliability-qualified NTL,3,2014-03-08,2014-05-06,221.759995,-73.050003,15,-106.919998


In [ ]:
ngcp_composites = four_day_composites(
    ngcp_load
)

ngcp_baseline = ngcp_composites.loc[
    ngcp_composites["block"] == -1,
    "value",
]

if ngcp_baseline.empty or pd.isna(ngcp_baseline.iloc[0]):
    raise ValueError(
        "The NGCP pre-Haiyan four-day baseline is missing."
    )

ngcp_load0 = float(
    ngcp_baseline.iloc[0]
)

ngcp_composites["recovery_pct"] = (
    100.0
    * ngcp_composites["value"]
    / ngcp_load0
)

ngcp_composites["method"] = (
    "NGCP 1 AM load"
)

comparison_trajectories = pd.concat(
    [
        roman_trajectory[
            [
                "block",
                "date_end",
                "recovery_pct",
                "method",
            ]
        ],
        rq_trajectory[
            [
                "block",
                "date_end",
                "recovery_pct",
                "method",
            ]
        ],
        ngcp_composites[
            [
                "block",
                "date_end",
                "recovery_pct",
                "method",
            ]
        ],
    ],
    ignore_index=True,
)

In [ ]:
fig = px.line(
    comparison_trajectories,
    x="date_end",
    y="recovery_pct",
    color="method",
    markers=True,
    color_discrete_map=SERIES_COLORS,
    category_orders={
        "method": [
            "Román-style NTL",
            "Reliability-qualified NTL",
            "NGCP 1 AM load",
        ]
    },
)

fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6),
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Composite end: %{x|%d %b %Y}<br>"
        "Recovery: %{y:.1f}%"
        "<extra></extra>"
    ),
)

fig.add_hline(
    y=100,
    line_width=1.5,
    line_dash="dot",
    line_color="#475569",
)

fig.add_vline(
    x=EVENT_DATE,
    line_width=2,
    line_dash="dash",
    line_color=EVENT_LINE_COLOR,
)

for boundary in [60, 120]:
    fig.add_vline(
        x=EVENT_DATE + pd.Timedelta(days=boundary),
        line_width=1.2,
        line_dash="dot",
        line_color="#94A3B8",
    )

fig.add_annotation(
    x=EVENT_DATE,
    y=1.02,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        size=15,
        color=EVENT_LINE_COLOR,
    ),
)

fig.update_layout(
    title=(
        "Román recovery trajectories and "
        "Samar–Leyte NGCP 1 AM load"
    ),
    template=PLOT_TEMPLATE,
    paper_bgcolor="rgba(0,0,0,0)",
    width=1200,
    height=650,
    margin=dict(
        l=80,
        r=40,
        t=120,
        b=80,
    ),
    font=dict(size=16),
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="left",
        x=0,
    ),
    xaxis_title="Four-day composite period",
    yaxis_title="Recovery relative to pre-Haiyan baseline (%)",
)

fig.show()

In [ ]:
# ============================================================
# DIAGNOSE THE EXTREME ROMÁN PROFILE
# ============================================================

baseline_diagnostics = pd.DataFrame(
    {
        "method": [
            "Román-style NTL",
            "Reliability-qualified NTL",
        ],
        "NTL0": [
            roman_composites.loc[
                roman_composites["block"] == -1,
                "value",
            ].iloc[0],
            rq_composites.loc[
                rq_composites["block"] == -1,
                "value",
            ].iloc[0],
        ],
        "baseline_observed_days": [
            roman_composites.loc[
                roman_composites["block"] == -1,
                "observed_days",
            ].iloc[0],
            rq_composites.loc[
                rq_composites["block"] == -1,
                "observed_days",
            ].iloc[0],
        ],
        "maximum_post_event_composite": [
            roman_composites.loc[
                roman_composites["block"] >= 0,
                "value",
            ].max(),
            rq_composites.loc[
                rq_composites["block"] >= 0,
                "value",
            ].max(),
        ],
        "maximum_recovery_pct": [
            roman_trajectory.loc[
                roman_trajectory["block"] >= 0,
                "recovery_pct",
            ].max(),
            rq_trajectory.loc[
                rq_trajectory["block"] >= 0,
                "recovery_pct",
            ].max(),
        ],
    }
)

display(
    baseline_diagnostics.round(3)
)

print("DNB attributes:")
display(dnb.attrs)

print(
    "Minimum DNB:",
    float(dnb.min(skipna=True).compute()),
)

print(
    "Maximum DNB:",
    float(dnb.max(skipna=True).compute()),
)

print("\nLargest Román-style composites:")

display(
    roman_trajectory.loc[
        roman_trajectory["block"] >= 0,
        [
            "block",
            "date_start",
            "date_end",
            "value",
            "observed_days",
            "recovery_pct",
        ],
    ]
    .nlargest(10, "recovery_pct")
    .round(3)
)

,method,NTL0,baseline_observed_days,maximum_post_event_composite,maximum_recovery_pct
0,Román-style NTL,0.203,3,11.978,5886.669922
1,Reliability-qualified NTL,0.553,2,1.689,305.446991


DNB attributes:


{'grid_mapping': 'spatial_ref',
 'scaling_applied': False,
 'source_descriptions': ['2013_05_12_DNB_BRDF_Corrected_NTL',
  '2013_05_13_DNB_BRDF_Corrected_NTL',
  '2013_05_14_DNB_BRDF_Corrected_NTL',
  '2013_05_15_DNB_BRDF_Corrected_NTL',
  '2013_05_16_DNB_BRDF_Corrected_NTL',
  '2013_05_17_DNB_BRDF_Corrected_NTL',
  '2013_05_18_DNB_BRDF_Corrected_NTL',
  '2013_05_19_DNB_BRDF_Corrected_NTL',
  '2013_05_20_DNB_BRDF_Corrected_NTL',
  '2013_05_21_DNB_BRDF_Corrected_NTL',
  '2013_05_22_DNB_BRDF_Corrected_NTL',
  '2013_05_23_DNB_BRDF_Corrected_NTL',
  '2013_05_24_DNB_BRDF_Corrected_NTL',
  '2013_05_25_DNB_BRDF_Corrected_NTL',
  '2013_05_26_DNB_BRDF_Corrected_NTL',
  '2013_05_27_DNB_BRDF_Corrected_NTL',
  '2013_05_28_DNB_BRDF_Corrected_NTL',
  '2013_05_29_DNB_BRDF_Corrected_NTL',
  '2013_05_30_DNB_BRDF_Corrected_NTL',
  '2013_05_31_DNB_BRDF_Corrected_NTL',
  '2013_06_01_DNB_BRDF_Corrected_NTL',
  '2013_06_02_DNB_BRDF_Corrected_NTL',
  '2013_06_03_DNB_BRDF_Corrected_NTL',
  '2013_06_04_DNB_BRD

Minimum DNB: 0.0
Maximum DNB: 361.5819396972656

Largest Román-style composites:


,block,date_start,date_end,value,observed_days,recovery_pct
40,39,2014-04-13,2014-04-16,11.978,3,5886.669922
18,17,2014-01-15,2014-01-18,3.458,1,1699.464966
3,2,2013-11-16,2013-11-19,1.851,3,909.481995
32,31,2014-03-12,2014-03-15,1.553,4,763.064026
10,9,2013-12-14,2013-12-17,1.450,4,712.586975
41,40,2014-04-17,2014-04-20,1.247,4,612.831970
19,18,2014-01-19,2014-01-22,0.871,2,428.149994
24,23,2014-02-08,2014-02-11,0.847,4,416.109009
42,41,2014-04-21,2014-04-24,0.840,4,412.773010
34,33,2014-03-20,2014-03-23,0.772,4,379.484985


In [ ]:
comparison_wide = (
    comparison_trajectories[
        comparison_trajectories["block"] >= 0
    ]
    .pivot(
        index="block",
        columns="method",
        values="recovery_pct",
    )
    .reset_index()
)

alignment_rows = []

for ntl_method in [
    "Román-style NTL",
    "Reliability-qualified NTL",
]:
    paired = comparison_wide[
        [
            ntl_method,
            "NGCP 1 AM load",
        ]
    ].dropna()

    if len(paired) < 3:
        alignment_rows.append(
            {
                "method": ntl_method,
                "n": len(paired),
                "pearson_r": np.nan,
                "spearman_rho": np.nan,
                "rmse_percentage_points": np.nan,
                "mae_percentage_points": np.nan,
            }
        )

        continue

    difference = (
        paired[ntl_method]
        - paired["NGCP 1 AM load"]
    )

    alignment_rows.append(
        {
            "method": ntl_method,
            "n": len(paired),
            "pearson_r": paired[
                ntl_method
            ].corr(
                paired["NGCP 1 AM load"],
                method="pearson",
            ),
            "spearman_rho": paired[
                ntl_method
            ].corr(
                paired["NGCP 1 AM load"],
                method="spearman",
            ),
            "rmse_percentage_points": np.sqrt(
                np.mean(difference**2)
            ),
            "mae_percentage_points": np.mean(
                np.abs(difference)
            ),
        }
    )

alignment_results = pd.DataFrame(
    alignment_rows
)

display(
    alignment_results.round(3)
)

,method,n,pearson_r,spearman_rho,rmse_percentage_points,mae_percentage_points
0,Román-style NTL,44,0.157,0.388,951.453,357.512
1,Reliability-qualified NTL,40,0.840,0.841,125.211,114.441


In [ ]:
roman_summary = (
    stage_results.groupby(
        "method",
        as_index=False,
    )
    .agg(
        stage_1_recovery_pct=(
            "recovery_pct",
            lambda values: values.iloc[0],
        ),
        stage_2_recovery_pct=(
            "recovery_pct",
            lambda values: values.iloc[1],
        ),
        stage_3_recovery_pct=(
            "recovery_pct",
            lambda values: values.iloc[2],
        ),
        total_ndwe_days=(
            "ndwe_days",
            "sum",
        ),
        available_stage_composites=(
            "available_4day_composites",
            "sum",
        ),
    )
)

roman_summary = roman_summary.merge(
    alignment_results,
    on="method",
    how="left",
)

display(
    roman_summary.round(2)
)

,method,stage_1_recovery_pct,stage_2_recovery_pct,stage_3_recovery_pct,total_ndwe_days,available_stage_composites,n,pearson_r,spearman_rho,rmse_percentage_points,mae_percentage_points
0,Reliability-qualified NTL,93.150002,163.279999,221.759995,-106.919998,40,40,0.84,0.84,125.21,114.44
1,Román-style NTL,225.949997,272.899994,697.919983,-538.059998,44,44,0.16,0.39,951.45,357.51


# Transferability of the Román et al. recovery metrics to Samar–Leyte

This notebook evaluates whether the electricity-recovery metrics developed by Román et al. (2019) can be transferred to the Samar–Leyte Haiyan case.

The comparison includes:

1. gap-filled VNP46A2 over baseline-lit pixels;
2. direct DNB-BRDF observations with `MQF == 0`;
3. gap-filled VNP46A2 over fixed GHSL G7 settlement support;
4. reliability-qualified DNB-BRDF with G7 and spatial coverage ≥50%;
5. reliability-qualified DNB-BRDF with G7 and spatial coverage ≥60%.

The notebook runs independently from the processed VNP46A1 and VNP46A2 Zarr stores and stops after the Román et al. replication.

In [ ]:
# ============================================================
# 19. TOTAL 180-DAY NDWE
# ============================================================

roman_total_results["signal"] = (
    pd.Categorical(
        roman_total_results["signal"],
        categories=SIGNAL_ORDER,
        ordered=True,
    )
)

roman_total_plot = (
    roman_total_results
    .sort_values("signal")
)

fig_roman_ndwe = px.bar(
    roman_total_plot,
    x="mean_total_ndwe_days",
    y="signal",
    color="signal",
    orientation="h",
    color_discrete_map=SIGNAL_COLORS,
    category_orders={
        "signal": SIGNAL_ORDER,
    },
    custom_data=[
        "baseline_support_pct",
        "total_valid_pixel_pct",
        "status",
    ],
)

fig_roman_ndwe.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Mean NDWE: %{x:.1f} days<br>"
        "Baseline support: "
        "%{customdata[0]:.1f}%<br>"
        "Complete metric support: "
        "%{customdata[1]:.1f}%<br>"
        "Status: %{customdata[2]}"
        "<extra></extra>"
    )
)

fig_roman_ndwe.update_layout(
    title=(
        "Román et al. net days without "
        "electricity: 180-day Haiyan period"
    ),
    template=PLOT_TEMPLATE,
    paper_bgcolor="rgba(0,0,0,0)",
    width=1200,
    height=600,
    margin=dict(
        l=270,
        r=40,
        t=100,
        b=80,
    ),
    font=dict(size=16),
    showlegend=False,
    xaxis_title=(
        "Mean pixel-level NDWE (days)"
    ),
    yaxis_title=None,
)

fig_roman_ndwe.show()

In [ ]:
# ============================================================
# 2. PATHS AND ANALYSIS SETTINGS
# ============================================================

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A1_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A1.zarr"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

GHSL_CANDIDATES = [
    VNP46_DIR / "GHSL_SMOD_E2015.tif",
    DATA_DIR / "ghsl" / "GHSL_SMOD_E2015.tif",
]

GHSL_PATH = next(
    (
        path
        for path in GHSL_CANDIDATES
        if path.exists()
    ),
    GHSL_CANDIDATES[0],
)

# Event and analysis period
EVENT_DATE = pd.Timestamp("2013-11-08")
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)

# Reliability settings
MQF_VALUES = (0,)
GHSL_CLASSES = (23, 30)
GHSL_GROUP_LABEL = "G7"

SC_THRESHOLD_50 = 50.0
SC_THRESHOLD_60 = 60.0

# Román et al. settings
ROMAN_BLOCK_DAYS = 4
ROMAN_STAGE_DAYS = 60
ROMAN_STAGE_COUNT = 3
ROMAN_BLOCKS_PER_STAGE = (
    ROMAN_STAGE_DAYS // ROMAN_BLOCK_DAYS
)
ROMAN_POST_DAYS = (
    ROMAN_STAGE_DAYS * ROMAN_STAGE_COUNT
)

# Plot settings
PLOT_TEMPLATE = "plotly_white"
EVENT_LINE_COLOR = "#2563EB"

SIGNAL_COLORS = {
    "Gap-filled | baseline-lit": "#D97706",
    "Direct MQF=0 | baseline-lit": "#64748B",
    "Gap-filled | G7": "#EA580C",
    "RQ DNB-BRDF | G7 | SC≥50%": "#059669",
    "RQ DNB-BRDF | G7 | SC≥60%": "#7C3AED",
}

SIGNAL_ORDER = list(SIGNAL_COLORS)

print("Project directory:", PROJECT_DIR)
print("VNP46A1:", A1_ZARR_PATH)
print("VNP46A2:", A2_ZARR_PATH)
print("GHSL:", GHSL_PATH)

Project directory: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery
VNP46A1: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/VNP46/processed/Haiyan_VNP46A1.zarr
VNP46A2: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/VNP46/processed/Haiyan_VNP46A2.zarr
GHSL: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/ghsl/GHSL_SMOD_E2015.tif


In [ ]:
# ============================================================
# 3. INPUT CHECKS
# ============================================================

required_paths = {
    "VNP46A1 Zarr": A1_ZARR_PATH,
    "VNP46A2 Zarr": A2_ZARR_PATH,
    "GHSL raster": GHSL_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required inputs were not found:\n"
        + "\n".join(missing_paths)
    )

print("All required inputs were found.")

All required inputs were found.


In [ ]:
# ============================================================
# 4. OPEN AND STANDARDISE THE ZARR STORES
# ============================================================

def open_zarr_safely(path):
    """Open Zarr with a non-Dask fallback."""
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            decode_cf=True,
            mask_and_scale=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            decode_cf=True,
            mask_and_scale=True,
        )


def standardise_date_dimension(ds):
    """
    Convert the stored observation axis to a `date` dimension.

    The Haiyan Zarr stores use a `date` coordinate associated with
    the original `processed` observation dimension.
    """
    if "date" not in ds.variables:
        raise KeyError(
            "No `date` variable was found. "
            f"Available variables: {list(ds.variables)}"
        )

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dimension = ds["date"].dims[0]

    dates = pd.DatetimeIndex(
        pd.to_datetime(ds["date"].values)
    ).normalize()

    if dates.hasnans:
        raise ValueError(
            "The Zarr store contains invalid date values."
        )

    if dates.has_duplicates:
        duplicate_dates = dates[
            dates.duplicated()
        ].unique()

        raise ValueError(
            "Duplicate dates were found: "
            f"{list(duplicate_dates[:10])}"
        )

    ds = ds.assign_coords(
        date=(
            observation_dimension,
            dates.values,
        )
    )

    if observation_dimension != "date":
        ds = ds.swap_dims(
            {observation_dimension: "date"}
        )

    return ds.sortby("date")


def prepare_spatial_dataset(ds):
    """Attach spatial dimensions and recover the stored CRS."""
    if "x" not in ds.dims or "y" not in ds.dims:
        raise ValueError(
            "The Zarr store must contain x and y dimensions."
        )

    ds = ds.rio.set_spatial_dims(
        x_dim="x",
        y_dim="y",
        inplace=False,
    )

    if ds.rio.crs is None:
        if "spatial_ref" not in ds.variables:
            raise ValueError(
                "No CRS or spatial_ref variable was found."
            )

        spatial_attributes = ds[
            "spatial_ref"
        ].attrs

        stored_crs = (
            spatial_attributes.get("crs_wkt")
            or spatial_attributes.get("spatial_ref")
        )

        if stored_crs is None:
            raise ValueError(
                "The CRS could not be recovered from spatial_ref."
            )

        ds = ds.rio.write_crs(
            stored_crs,
            inplace=False,
        )

    return ds


a1 = prepare_spatial_dataset(
    standardise_date_dimension(
        open_zarr_safely(A1_ZARR_PATH)
    )
)

a2 = prepare_spatial_dataset(
    standardise_date_dimension(
        open_zarr_safely(A2_ZARR_PATH)
    )
)

a1 = a1.sel(
    date=slice(ANALYSIS_START, ANALYSIS_END)
)

a2 = a2.sel(
    date=slice(ANALYSIS_START, ANALYSIS_END)
)

print("A1 dimensions:", dict(a1.sizes))
print("A2 dimensions:", dict(a2.sizes))

A1 dimensions: {'date': 546, 'y': 674, 'x': 473}
A2 dimensions: {'date': 546, 'y': 674, 'x': 473}


In [ ]:
# ============================================================
# 5. EXTRACT THE REQUIRED VNP46A2 BANDS
# ============================================================

DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"

required_a2_bands = [
    DNB_BAND,
    GAP_BAND,
    MQF_BAND,
]

missing_a2_bands = [
    band
    for band in required_a2_bands
    if band not in a2.data_vars
]

if missing_a2_bands:
    raise KeyError(
        "Missing VNP46A2 bands: "
        f"{missing_a2_bands}\n"
        f"Available bands: {list(a2.data_vars)}"
    )

dnb = a2[DNB_BAND].astype("float32")
gap_filled = a2[GAP_BAND].astype("float32")
mqf = a2[MQF_BAND]

dnb, gap_filled, mqf = xr.align(
    dnb,
    gap_filled,
    mqf,
    join="inner",
)

SPATIAL_DIMS = ("y", "x")

print("DNB band:", DNB_BAND)
print("Gap-filled band:", GAP_BAND)
print("MQF band:", MQF_BAND)
print(
    "Date range:",
    pd.DatetimeIndex(dnb["date"].values)
    .min()
    .date(),
    "to",
    pd.DatetimeIndex(dnb["date"].values)
    .max()
    .date(),
)

DNB band: DNB_BRDF_Corrected_NTL
Gap-filled band: Gap_Filled_DNB_BRDF_Corrected_NTL
MQF band: Mandatory_Quality_Flag
Date range: 2013-05-12 to 2014-11-08


In [ ]:
# ============================================================
# 6. CONSTRUCT THE DIRECT MQF-QUALIFIED DNB SERIES
# ============================================================

mqf_valid = mqf.isin(MQF_VALUES)

fresh_dnb = dnb.where(mqf_valid)

print("Accepted MQF values:", MQF_VALUES)
print(
    "Total dates:",
    fresh_dnb.sizes["date"],
)

Accepted MQF values: (0,)
Total dates: 546


In [ ]:
# ============================================================
# 7. REPROJECT GHSL TO THE VIIRS GRID
# ============================================================

ghsl = rxr.open_rasterio(
    GHSL_PATH,
    masked=True,
)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(
        band=0,
        drop=True,
    )

if ghsl.rio.crs is None:
    raise ValueError(
        "The GHSL raster does not contain a readable CRS."
    )

viirs_template = dnb.isel(
    date=0,
    drop=True,
)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
)

ghsl_viirs = ghsl_viirs.assign_coords(
    x=viirs_template["x"],
    y=viirs_template["y"],
)

ghsl_viirs = ghsl_viirs.rename(
    "GHSL_SMOD"
)

g7_mask = ghsl_viirs.isin(
    GHSL_CLASSES
).fillna(False)

print(
    f"GHSL {GHSL_GROUP_LABEL} classes:",
    GHSL_CLASSES,
)

GHSL G7 classes: (23, 30)


In [ ]:
# ============================================================
# 8. BUILD THE FIXED PRE-HAIYAN SIGNAL MASKS
# ============================================================

BASELINE_END = EVENT_DATE - pd.Timedelta(days=1)

pre_event_fresh = fresh_dnb.sel(
    date=slice(
        ANALYSIS_START,
        BASELINE_END,
    )
)

pre_event_valid_days = (
    pre_event_fresh
    .notnull()
    .sum(dim="date")
)

pre_event_mean = pre_event_fresh.mean(
    dim="date",
    skipna=True,
)

baseline_lit_mask = (
    (pre_event_valid_days >= 1)
    & np.isfinite(pre_event_mean)
    & (pre_event_mean > 0)
).fillna(False)

fixed_g7_mask = (
    baseline_lit_mask
    & g7_mask
).fillna(False)


def scalar_int(value):
    """Convert a scalar xarray result to an integer."""
    if hasattr(value, "compute"):
        value = value.compute()

    if hasattr(value, "values"):
        value = value.values

    return int(
        np.asarray(value).reshape(-1)[0]
    )


mask_summary = pd.DataFrame(
    {
        "mask": [
            "Complete VIIRS grid",
            "GHSL G7",
            "Pre-Haiyan baseline-lit",
            "Fixed G7 + pre-Haiyan support",
        ],
        "pixel_count": [
            (
                viirs_template.sizes["y"]
                * viirs_template.sizes["x"]
            ),
            scalar_int(g7_mask.sum()),
            scalar_int(baseline_lit_mask.sum()),
            scalar_int(fixed_g7_mask.sum()),
        ],
    }
)

display(mask_summary)

if scalar_int(fixed_g7_mask.sum()) == 0:
    raise ValueError(
        "The fixed G7 signal mask contains no pixels. "
        "Inspect the GHSL alignment and class settings."
    )

,mask,pixel_count
0,Complete VIIRS grid,318802
1,GHSL G7,2682
2,Pre-Haiyan baseline-lit,103801
3,Fixed G7 + pre-Haiyan support,2585


In [ ]:
# ============================================================
# 9. DAILY SPATIAL COVERAGE
# ============================================================

def calculate_spatial_coverage(mask):
    """Calculate daily MQF-qualified spatial coverage."""
    mask_pixel_count = scalar_int(mask.sum())

    if mask_pixel_count == 0:
        raise ValueError(
            "Cannot calculate coverage for an empty mask."
        )

    qualified_pixel_count = (
        fresh_dnb
        .notnull()
        .where(mask, False)
        .sum(dim=SPATIAL_DIMS)
    )

    spatial_coverage = (
        100.0
        * qualified_pixel_count
        / mask_pixel_count
    )

    return spatial_coverage


baseline_lit_coverage = (
    calculate_spatial_coverage(
        baseline_lit_mask
    )
)

g7_coverage = calculate_spatial_coverage(
    fixed_g7_mask
)

In [ ]:
# ============================================================
# 10. CONSTRUCT THE FIVE ANALYSIS SIGNALS
# ============================================================

gap_baseline_lit = gap_filled.where(
    baseline_lit_mask
)

direct_baseline_lit = fresh_dnb.where(
    baseline_lit_mask
)

gap_g7 = gap_filled.where(
    fixed_g7_mask
)

direct_g7 = fresh_dnb.where(
    fixed_g7_mask
)

rq_g7_50 = direct_g7.where(
    g7_coverage >= SC_THRESHOLD_50
)

rq_g7_60 = direct_g7.where(
    g7_coverage >= SC_THRESHOLD_60
)

signal_cubes = {
    "Gap-filled | baseline-lit": gap_baseline_lit,
    "Direct MQF=0 | baseline-lit": direct_baseline_lit,
    "Gap-filled | G7": gap_g7,
    "RQ DNB-BRDF | G7 | SC≥50%": rq_g7_50,
    "RQ DNB-BRDF | G7 | SC≥60%": rq_g7_60,
}

signal_masks = {
    "Gap-filled | baseline-lit": baseline_lit_mask,
    "Direct MQF=0 | baseline-lit": baseline_lit_mask,
    "Gap-filled | G7": fixed_g7_mask,
    "RQ DNB-BRDF | G7 | SC≥50%": fixed_g7_mask,
    "RQ DNB-BRDF | G7 | SC≥60%": fixed_g7_mask,
}

signal_coverages = {
    "Gap-filled | baseline-lit": baseline_lit_coverage,
    "Direct MQF=0 | baseline-lit": baseline_lit_coverage,
    "Gap-filled | G7": g7_coverage,
    "RQ DNB-BRDF | G7 | SC≥50%": g7_coverage,
    "RQ DNB-BRDF | G7 | SC≥60%": g7_coverage,
}

print("Analysis signals:")
for signal_name in SIGNAL_ORDER:
    print(" -", signal_name)

Analysis signals:
 - Gap-filled | baseline-lit
 - Direct MQF=0 | baseline-lit
 - Gap-filled | G7
 - RQ DNB-BRDF | G7 | SC≥50%
 - RQ DNB-BRDF | G7 | SC≥60%


In [ ]:
# ============================================================
# 11. BUILD REGIONAL DAILY TIME SERIES
# ============================================================

daily_signal_frames = []

for signal_name in SIGNAL_ORDER:
    regional_ntl = signal_cubes[
        signal_name
    ].mean(
        dim=SPATIAL_DIMS,
        skipna=True,
    )

    daily_frame = xr.Dataset(
        {
            "ntl": regional_ntl,
            "spatial_coverage_pct": (
                signal_coverages[signal_name]
            ),
        }
    ).to_dataframe().reset_index()

    daily_frame["date"] = pd.to_datetime(
        daily_frame["date"]
    )

    daily_frame["signal"] = signal_name

    daily_signal_frames.append(
        daily_frame
    )

daily_signals = pd.concat(
    daily_signal_frames,
    ignore_index=True,
)

daily_signals["signal"] = pd.Categorical(
    daily_signals["signal"],
    categories=SIGNAL_ORDER,
    ordered=True,
)

daily_support_summary = (
    daily_signals
    .groupby(
        "signal",
        observed=True,
    )
    .agg(
        available_ntl_dates=("ntl", "count"),
        total_dates=("date", "size"),
        median_spatial_coverage=(
            "spatial_coverage_pct",
            "median",
        ),
        minimum_spatial_coverage=(
            "spatial_coverage_pct",
            "min",
        ),
        maximum_spatial_coverage=(
            "spatial_coverage_pct",
            "max",
        ),
    )
    .reset_index()
)

display(
    daily_support_summary.round(2)
)

,signal,available_ntl_dates,total_dates,median_spatial_coverage,minimum_spatial_coverage,maximum_spatial_coverage
0,Gap-filled | baseline-lit,544,546,17.77,0.0,98.94
1,Direct MQF=0 | baseline-lit,434,546,17.77,0.0,98.94
2,Gap-filled | G7,542,546,12.73,0.0,99.15
3,RQ DNB-BRDF | G7 | SC≥50%,153,546,12.73,0.0,99.15
4,RQ DNB-BRDF | G7 | SC≥60%,120,546,12.73,0.0,99.15


In [ ]:
# ============================================================
# 12. ROMÁN STAGE DEFINITIONS
# ============================================================

ROMAN_BASELINE_START = (
    EVENT_DATE
    - pd.Timedelta(days=ROMAN_BLOCK_DAYS)
)

ROMAN_BASELINE_END = (
    EVENT_DATE
    - pd.Timedelta(days=1)
)

ROMAN_ANALYSIS_END = (
    EVENT_DATE
    + pd.Timedelta(
        days=ROMAN_POST_DAYS - 1
    )
)

roman_stage_definitions = pd.DataFrame(
    {
        "stage": [
            1,
            2,
            3,
        ],
        "start_date": [
            EVENT_DATE,
            EVENT_DATE
            + pd.Timedelta(days=60),
            EVENT_DATE
            + pd.Timedelta(days=120),
        ],
        "end_date": [
            EVENT_DATE
            + pd.Timedelta(days=59),
            EVENT_DATE
            + pd.Timedelta(days=119),
            EVENT_DATE
            + pd.Timedelta(days=179),
        ],
        "duration_days": [
            60,
            60,
            60,
        ],
        "four_day_composites": [
            15,
            15,
            15,
        ],
    }
)

print(
    "NTL₀ baseline:",
    ROMAN_BASELINE_START.date(),
    "to",
    ROMAN_BASELINE_END.date(),
)

display(roman_stage_definitions)

NTL₀ baseline: 2013-11-04 to 2013-11-07


,stage,start_date,end_date,duration_days,four_day_composites
0,1,2013-11-08,2014-01-06,60,15
1,2,2014-01-07,2014-03-07,60,15
2,3,2014-03-08,2014-05-06,60,15


In [ ]:
# ============================================================
# 13. EVENT-ANCHORED FOUR-DAY COMPOSITES
# ============================================================

def build_roman_composites(signal_cube):
    """
    Aggregate daily observations into four-day blocks anchored
    on the Haiyan event date.

    Block -1 = four days immediately before Haiyan.
    Block  0 = Haiyan date through post-event day 3.
    Blocks 0-44 = three 60-day recovery stages.
    """
    signal_window = signal_cube.sel(
        date=slice(
            ROMAN_BASELINE_START,
            ROMAN_ANALYSIS_END,
        )
    )

    dates = pd.DatetimeIndex(
        pd.to_datetime(
            signal_window["date"].values
        )
    ).normalize()

    relative_days = (
        (dates - EVENT_DATE)
        / pd.Timedelta(days=1)
    ).astype("int64")

    roman_blocks = np.floor_divide(
        relative_days,
        ROMAN_BLOCK_DAYS,
    ).astype("int64")

    signal_window = signal_window.assign_coords(
        roman_block=(
            "date",
            roman_blocks,
        )
    )

    composites = (
        signal_window
        .groupby("roman_block")
        .mean(
            dim="date",
            skipna=True,
        )
    )

    valid_days = (
        signal_window
        .notnull()
        .groupby("roman_block")
        .sum(dim="date")
    )

    expected_blocks = np.arange(
        -1,
        (
            ROMAN_STAGE_COUNT
            * ROMAN_BLOCKS_PER_STAGE
        ),
    )

    composites = composites.reindex(
        roman_block=expected_blocks
    )

    valid_days = valid_days.reindex(
        roman_block=expected_blocks,
        fill_value=0,
    )

    composites = composites.where(
        valid_days >= 1
    )

    return composites, valid_days

In [ ]:
# ============================================================
# 14. ROMÁN PIXEL-LEVEL METRICS
# ============================================================

def calculate_roman_metrics(
    signal_cube,
):
    """
    Calculate pixel-level Román %Recovery and NDWE.
    """
    composites, valid_days = (
        build_roman_composites(
            signal_cube
        )
    )

    baseline_ntl = composites.sel(
        roman_block=-1,
        drop=True,
    )

    baseline_valid_days = valid_days.sel(
        roman_block=-1,
        drop=True,
    )

    baseline_supported = (
        np.isfinite(baseline_ntl)
        & (baseline_ntl > 0)
    )

    baseline_ntl = baseline_ntl.where(
        baseline_supported
    )

    baseline_valid_days = (
        baseline_valid_days.where(
            baseline_supported
        )
    )

    stage_ntl_list = []
    recovery_list = []
    ndwe_list = []
    observed_composite_list = []

    for stage in range(
        1,
        ROMAN_STAGE_COUNT + 1,
    ):
        first_block = (
            (stage - 1)
            * ROMAN_BLOCKS_PER_STAGE
        )

        last_block = (
            stage
            * ROMAN_BLOCKS_PER_STAGE
            - 1
        )

        stage_composites = composites.sel(
            roman_block=slice(
                first_block,
                last_block,
            )
        )

        stage_ntl = stage_composites.mean(
            dim="roman_block",
            skipna=True,
        )

        observed_composites = (
            stage_composites
            .notnull()
            .sum(dim="roman_block")
        )

        recovery_pct = (
            100.0
            * stage_ntl
            / baseline_ntl
        )

        recovery_pct = recovery_pct.where(
            np.isfinite(recovery_pct)
        )

        # Original Román equation:
        # no clipping is applied when recovery exceeds 100%.
        ndwe_stage_days = (
            1.0
            - recovery_pct / 100.0
        ) * ROMAN_STAGE_DAYS

        stage_ntl_list.append(
            stage_ntl
        )

        recovery_list.append(
            recovery_pct
        )

        ndwe_list.append(
            ndwe_stage_days
        )

        observed_composite_list.append(
            observed_composites
        )

    stage_coordinate = pd.Index(
        [1, 2, 3],
        name="stage",
    )

    stage_ntl = xr.concat(
        stage_ntl_list,
        dim=stage_coordinate,
    )

    recovery_pct = xr.concat(
        recovery_list,
        dim=stage_coordinate,
    )

    ndwe_stage_days = xr.concat(
        ndwe_list,
        dim=stage_coordinate,
    )

    observed_composites = xr.concat(
        observed_composite_list,
        dim=stage_coordinate,
    )

    total_ndwe_days = ndwe_stage_days.sum(
        dim="stage",
        skipna=False,
    )

    pixel_metrics = xr.Dataset(
        {
            "baseline_ntl": baseline_ntl,
            "baseline_valid_days": (
                baseline_valid_days
            ),
            "stage_ntl": stage_ntl,
            "recovery_pct": recovery_pct,
            "ndwe_stage_days": (
                ndwe_stage_days
            ),
            "observed_composites": (
                observed_composites
            ),
            "total_ndwe_days": (
                total_ndwe_days
            ),
        }
    ).compute()

    # Four-day pixel-wise recovery trajectory
    composite_recovery = (
        100.0
        * composites
        / pixel_metrics["baseline_ntl"]
    )

    trajectory = xr.Dataset(
        {
            "recovery_pct": (
                composite_recovery.mean(
                    dim=SPATIAL_DIMS,
                    skipna=True,
                )
            ),
            "supported_pixels": (
                composite_recovery
                .notnull()
                .sum(dim=SPATIAL_DIMS)
            ),
        }
    ).compute()

    return pixel_metrics, trajectory

In [ ]:
# ============================================================
# 15. RUN THE ROMÁN REPLICATION
# ============================================================

def scalar_float(value):
    """Convert a scalar xarray result to float."""
    if hasattr(value, "compute"):
        value = value.compute()

    if hasattr(value, "values"):
        value = value.values

    array = np.asarray(value)

    if array.size == 0:
        return np.nan

    return float(
        array.reshape(-1)[0]
    )


def count_valid_pixels(values):
    """Count finite values in a spatial array."""
    valid = np.isfinite(values)

    return scalar_int(
        valid.astype("int32").sum()
    )


roman_pixel_results = {}
roman_stage_rows = []
roman_total_rows = []
roman_trajectory_frames = []

for signal_name in SIGNAL_ORDER:
    print(
        "Processing:",
        signal_name,
    )

    pixel_metrics, trajectory = (
        calculate_roman_metrics(
            signal_cubes[signal_name]
        )
    )

    roman_pixel_results[
        signal_name
    ] = pixel_metrics

    mask_pixel_count = scalar_int(
        signal_masks[signal_name].sum()
    )

    baseline_supported_pixels = (
        count_valid_pixels(
            pixel_metrics["baseline_ntl"]
        )
    )

    baseline_support_pct = (
        100.0
        * baseline_supported_pixels
        / mask_pixel_count
    )

    # --------------------------------------------------------
    # Stage summaries
    # --------------------------------------------------------

    for stage in range(
        1,
        ROMAN_STAGE_COUNT + 1,
    ):
        stage_data = pixel_metrics.sel(
            stage=stage
        )

        stage_valid_pixels = (
            count_valid_pixels(
                stage_data["recovery_pct"]
            )
        )

        roman_stage_rows.append(
            {
                "signal": signal_name,
                "stage": stage,
                "stage_label": (
                    f"Stage {stage}"
                ),
                "start_date": (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=(
                            stage - 1
                        ) * ROMAN_STAGE_DAYS
                    )
                ),
                "end_date": (
                    EVENT_DATE
                    + pd.Timedelta(
                        days=(
                            stage
                            * ROMAN_STAGE_DAYS
                        ) - 1
                    )
                ),
                "mean_recovery_pct": (
                    scalar_float(
                        stage_data[
                            "recovery_pct"
                        ].mean(
                            dim=SPATIAL_DIMS,
                            skipna=True,
                        )
                    )
                ),
                "mean_ndwe_days": (
                    scalar_float(
                        stage_data[
                            "ndwe_stage_days"
                        ].mean(
                            dim=SPATIAL_DIMS,
                            skipna=True,
                        )
                    )
                ),
                "mean_observed_composites": (
                    scalar_float(
                        stage_data[
                            "observed_composites"
                        ].where(
                            np.isfinite(
                                stage_data[
                                    "recovery_pct"
                                ]
                            )
                        ).mean(
                            dim=SPATIAL_DIMS,
                            skipna=True,
                        )
                    )
                ),
                "expected_composites": (
                    ROMAN_BLOCKS_PER_STAGE
                ),
                "baseline_support_pct": (
                    baseline_support_pct
                ),
                "stage_valid_pixel_pct": (
                    100.0
                    * stage_valid_pixels
                    / mask_pixel_count
                ),
            }
        )

    # --------------------------------------------------------
    # Total NDWE
    # --------------------------------------------------------

    total_valid_pixels = count_valid_pixels(
        pixel_metrics["total_ndwe_days"]
    )

    roman_total_rows.append(
        {
            "signal": signal_name,
            "analysis_days": (
                ROMAN_POST_DAYS
            ),
            "mean_total_ndwe_days": (
                scalar_float(
                    pixel_metrics[
                        "total_ndwe_days"
                    ].mean(
                        dim=SPATIAL_DIMS,
                        skipna=True,
                    )
                )
            ),
            "baseline_supported_pixels": (
                baseline_supported_pixels
            ),
            "baseline_support_pct": (
                baseline_support_pct
            ),
            "total_valid_pixels": (
                total_valid_pixels
            ),
            "total_valid_pixel_pct": (
                100.0
                * total_valid_pixels
                / mask_pixel_count
            ),
            "status": (
                "Estimated"
                if total_valid_pixels > 0
                else "Not estimable"
            ),
        }
    )

    # --------------------------------------------------------
    # Four-day trajectory
    # --------------------------------------------------------

    trajectory_frame = (
        trajectory
        .to_dataframe()
        .reset_index()
    )

    trajectory_frame["date"] = (
        EVENT_DATE
        + pd.to_timedelta(
            (
                trajectory_frame[
                    "roman_block"
                ]
                * ROMAN_BLOCK_DAYS
            )
            + (
                ROMAN_BLOCK_DAYS - 1
            ) / 2,
            unit="D",
        )
    )

    trajectory_frame["signal"] = (
        signal_name
    )

    roman_trajectory_frames.append(
        trajectory_frame
    )


roman_stage_results = pd.DataFrame(
    roman_stage_rows
)

roman_total_results = pd.DataFrame(
    roman_total_rows
)

roman_trajectories = pd.concat(
    roman_trajectory_frames,
    ignore_index=True,
)

Processing: Gap-filled | baseline-lit
Processing: Direct MQF=0 | baseline-lit
Processing: Gap-filled | G7
Processing: RQ DNB-BRDF | G7 | SC≥50%
Processing: RQ DNB-BRDF | G7 | SC≥60%


In [ ]:
# ============================================================
# 16. ROMÁN RESULTS TABLES
# ============================================================

roman_stage_results[
    "composite_support_pct"
] = (
    100.0
    * roman_stage_results[
        "mean_observed_composites"
    ]
    / roman_stage_results[
        "expected_composites"
    ]
)

roman_stage_display = (
    roman_stage_results.copy()
)

roman_stage_display[
    [
        "mean_recovery_pct",
        "mean_ndwe_days",
        "mean_observed_composites",
        "composite_support_pct",
        "baseline_support_pct",
        "stage_valid_pixel_pct",
    ]
] = roman_stage_display[
    [
        "mean_recovery_pct",
        "mean_ndwe_days",
        "mean_observed_composites",
        "composite_support_pct",
        "baseline_support_pct",
        "stage_valid_pixel_pct",
    ]
].round(2)

roman_total_display = (
    roman_total_results.copy()
)

roman_total_display[
    [
        "mean_total_ndwe_days",
        "baseline_support_pct",
        "total_valid_pixel_pct",
    ]
] = roman_total_display[
    [
        "mean_total_ndwe_days",
        "baseline_support_pct",
        "total_valid_pixel_pct",
    ]
].round(2)

display(
    roman_stage_display[
        [
            "signal",
            "stage_label",
            "start_date",
            "end_date",
            "mean_recovery_pct",
            "mean_ndwe_days",
            "mean_observed_composites",
            "composite_support_pct",
            "baseline_support_pct",
            "stage_valid_pixel_pct",
        ]
    ]
)

display(
    roman_total_display[
        [
            "signal",
            "analysis_days",
            "mean_total_ndwe_days",
            "baseline_supported_pixels",
            "baseline_support_pct",
            "total_valid_pixels",
            "total_valid_pixel_pct",
            "status",
        ]
    ]
)

,signal,stage_label,start_date,end_date,mean_recovery_pct,mean_ndwe_days,mean_observed_composites,composite_support_pct,baseline_support_pct,stage_valid_pixel_pct
0,Gap-filled | baseline-lit,Stage 1,2013-11-08,2014-01-06,98.01,1.19,15.00,100.00,100.00,100.00
1,Gap-filled | baseline-lit,Stage 2,2014-01-07,2014-03-07,101.09,-0.65,15.00,100.00,100.00,100.00
2,Gap-filled | baseline-lit,Stage 3,2014-03-08,2014-05-06,242.66,-85.60,15.00,100.00,100.00,100.00
3,Direct MQF=0 | baseline-lit,Stage 1,2013-11-08,2014-01-06,231.47,-78.88,11.72,78.12,42.04,42.04
4,Direct MQF=0 | baseline-lit,Stage 2,2014-01-07,2014-03-07,234.01,-80.40,10.16,67.74,42.04,42.04
5,Direct MQF=0 | baseline-lit,Stage 3,2014-03-08,2014-05-06,541.16,-264.70,11.70,77.97,42.04,42.04
6,Gap-filled | G7,Stage 1,2013-11-08,2014-01-06,70.09,17.95,15.00,100.00,100.00,100.00
7,Gap-filled | G7,Stage 2,2014-01-07,2014-03-07,81.62,11.03,15.00,100.00,100.00,100.00
8,Gap-filled | G7,Stage 3,2014-03-08,2014-05-06,147.30,-28.38,15.00,100.00,100.00,100.00
9,RQ DNB-BRDF | G7 | SC≥50%,Stage 1,2013-11-08,2014-01-06,NaN,NaN,NaN,NaN,0.00,0.00


,signal,analysis_days,mean_total_ndwe_days,baseline_supported_pixels,baseline_support_pct,total_valid_pixels,total_valid_pixel_pct,status
0,Gap-filled | baseline-lit,180,-85.06,103801,100.00,103801,100.00,Estimated
1,Direct MQF=0 | baseline-lit,180,-423.98,43634,42.04,43634,42.04,Estimated
2,Gap-filled | G7,180,0.59,2585,100.00,2585,100.00,Estimated
3,RQ DNB-BRDF | G7 | SC≥50%,180,NaN,0,0.00,0,0.00,Not estimable
4,RQ DNB-BRDF | G7 | SC≥60%,180,NaN,0,0.00,0,0.00,Not estimable


In [ ]:
# ============================================================
# 17. FOUR-DAY RECOVERY TRAJECTORIES
# ============================================================

roman_trajectories["signal"] = (
    pd.Categorical(
        roman_trajectories["signal"],
        categories=SIGNAL_ORDER,
        ordered=True,
    )
)

fig_roman_trajectory = px.line(
    roman_trajectories,
    x="date",
    y="recovery_pct",
    color="signal",
    markers=True,
    color_discrete_map=SIGNAL_COLORS,
    category_orders={
        "signal": SIGNAL_ORDER,
    },
    custom_data=[
        "supported_pixels",
    ],
)

fig_roman_trajectory.update_traces(
    line=dict(width=2.5),
    marker=dict(size=7),
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Date: %{x|%d %b %Y}<br>"
        "Recovery: %{y:.1f}%<br>"
        "Supported pixels: "
        "%{customdata[0]:,.0f}"
        "<extra></extra>"
    ),
)

fig_roman_trajectory.add_hline(
    y=100,
    line_width=1.5,
    line_dash="dot",
    line_color="#475569",
)

for boundary_day in [60, 120]:
    fig_roman_trajectory.add_vline(
        x=(
            EVENT_DATE
            + pd.Timedelta(
                days=boundary_day
            )
        ),
        line_width=1.2,
        line_dash="dot",
        line_color="#94A3B8",
    )

fig_roman_trajectory.add_vline(
    x=EVENT_DATE,
    line_width=2,
    line_dash="dash",
    line_color=EVENT_LINE_COLOR,
)

fig_roman_trajectory.add_annotation(
    x=EVENT_DATE,
    y=1.02,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        size=15,
        color=EVENT_LINE_COLOR,
    ),
)

fig_roman_trajectory.update_layout(
    title=(
        "Román et al. four-day recovery "
        "trajectories: Samar–Leyte"
    ),
    template=PLOT_TEMPLATE,
    paper_bgcolor="rgba(0,0,0,0)",
    width=1200,
    height=650,
    margin=dict(
        l=80,
        r=40,
        t=130,
        b=80,
    ),
    font=dict(size=16),
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="left",
        x=0,
    ),
    xaxis_title="Date",
    yaxis_title=(
        "Pixel-wise recovery relative "
        "to NTL₀ (%)"
    ),
)

fig_roman_trajectory.show()


In [ ]:
# ============================================================
# 18. STAGE-LEVEL RECOVERY
# ============================================================

roman_stage_results["signal"] = (
    pd.Categorical(
        roman_stage_results["signal"],
        categories=SIGNAL_ORDER,
        ordered=True,
    )
)

roman_stage_results["stage_label"] = (
    pd.Categorical(
        roman_stage_results[
            "stage_label"
        ],
        categories=[
            "Stage 1",
            "Stage 2",
            "Stage 3",
        ],
        ordered=True,
    )
)

fig_roman_stages = px.bar(
    roman_stage_results,
    x="stage_label",
    y="mean_recovery_pct",
    color="signal",
    barmode="group",
    color_discrete_map=SIGNAL_COLORS,
    category_orders={
        "signal": SIGNAL_ORDER,
        "stage_label": [
            "Stage 1",
            "Stage 2",
            "Stage 3",
        ],
    },
    custom_data=[
        "mean_ndwe_days",
        "composite_support_pct",
        "stage_valid_pixel_pct",
    ],
)

fig_roman_stages.update_traces(
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "%{x}<br>"
        "Recovery: %{y:.1f}%<br>"
        "NDWE contribution: "
        "%{customdata[0]:.1f} days<br>"
        "Composite support: "
        "%{customdata[1]:.1f}%<br>"
        "Valid pixels: "
        "%{customdata[2]:.1f}%"
        "<extra></extra>"
    )
)

fig_roman_stages.add_hline(
    y=100,
    line_width=1.5,
    line_dash="dot",
    line_color="#475569",
)

fig_roman_stages.update_layout(
    title=(
        "Román et al. stage-level "
        "recovery: Samar–Leyte"
    ),
    template=PLOT_TEMPLATE,
    paper_bgcolor="rgba(0,0,0,0)",
    width=1200,
    height=650,
    margin=dict(
        l=80,
        r=40,
        t=130,
        b=80,
    ),
    font=dict(size=16),
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="left",
        x=0,
    ),
    xaxis_title="Recovery stage",
    yaxis_title=(
        "Mean pixel-wise recovery (%)"
    ),
)

fig_roman_stages.show()

In [ ]:
# ============================================================
# 19. TOTAL 180-DAY NDWE
# ============================================================

roman_total_results["signal"] = (
    pd.Categorical(
        roman_total_results["signal"],
        categories=SIGNAL_ORDER,
        ordered=True,
    )
)

roman_total_plot = (
    roman_total_results
    .sort_values("signal")
)

fig_roman_ndwe = px.bar(
    roman_total_plot,
    x="mean_total_ndwe_days",
    y="signal",
    color="signal",
    orientation="h",
    color_discrete_map=SIGNAL_COLORS,
    category_orders={
        "signal": SIGNAL_ORDER,
    },
    custom_data=[
        "baseline_support_pct",
        "total_valid_pixel_pct",
        "status",
    ],
)

fig_roman_ndwe.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Mean NDWE: %{x:.1f} days<br>"
        "Baseline support: "
        "%{customdata[0]:.1f}%<br>"
        "Complete metric support: "
        "%{customdata[1]:.1f}%<br>"
        "Status: %{customdata[2]}"
        "<extra></extra>"
    )
)

fig_roman_ndwe.update_layout(
    title=(
        "Román et al. net days without "
        "electricity: 180-day Haiyan period"
    ),
    template=PLOT_TEMPLATE,
    paper_bgcolor="rgba(0,0,0,0)",
    width=1200,
    height=600,
    margin=dict(
        l=270,
        r=40,
        t=100,
        b=80,
    ),
    font=dict(size=16),
    showlegend=False,
    xaxis_title=(
        "Mean pixel-level NDWE (days)"
    ),
    yaxis_title=None,
)

fig_roman_ndwe.show()